# Late Fusion Bill Passage Classifier

PyTorch notebook for binary bill passage prediction using a late-fusion architecture that combines:
- a BERT text encoder for bill text
- categorical embeddings for metadata
- a linear projection for numeric metadata

The notebook follows the same parse-separately-then-merge pattern as the NLP preprocessing notebook.

Default metadata scope: intro-only features to reduce leakage risk.

Planned outputs:
- merged bill table with attached raw text
- tokenized dataset for late fusion training
- a PyTorch model with a Hugging Face BERT backbone
- imbalance-aware loss configuration

## 0. Setup and Dependencies

In [ ]:
import re
import subprocess
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
)
from torch import nn
from torch.utils.data import Dataset, random_split
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
NORM = DATA / 'normalized'
CA_TXT = DATA / 'canada_bill_text'
US_TXT = DATA / 'us_bill_text'

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 256
CHUNK_STRIDE = 64
MAX_CHUNKS = 3
BATCH_SIZE = 2
SEED = 42

# Optional installs for a fresh environment. Uncomment if needed.
# %pip install torch transformers accelerate scikit-learn tqdm

print(f'ROOT: {ROOT}')
print(f'MODEL_NAME: {MODEL_NAME}')
print(f'Memory-safe defaults: MAX_LENGTH={MAX_LENGTH}, CHUNK_STRIDE={CHUNK_STRIDE}, MAX_CHUNKS={MAX_CHUNKS}, BATCH_SIZE={BATCH_SIZE}')

ROOT: C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440
MODEL_NAME: nlpaueb/legal-bert-base-uncased
Memory-safe defaults: MAX_LENGTH=256, CHUNK_STRIDE=64, MAX_CHUNKS=3, BATCH_SIZE=2


## 1. Load and Merge the Bill Table

This stage mirrors the NLP notebook: load the normalized bill table, parse Canada and US texts separately, then attach a single raw-text column to the bill-level dataframe at the end.

In [2]:
bills_path = NORM / 'bills.csv'
if not bills_path.exists():
    print('bills.csv not found - running normalize.py ...')
    result = subprocess.run(['python', str(ROOT / 'scripts' / 'normalize.py')], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('normalize.py failed')

bills = pd.read_csv(bills_path, low_memory=False)
print(f'Loaded {len(bills):,} rows from bills.csv')
print(f'Sources: {bills["source"].value_counts().to_dict()}')

# Intro-only metadata defaults; switch to the full normalized list if you want a leakage-prone comparison.
NUMERIC_COLS = [
    'year',
    'title_word_count',
    'description_word_count',
    'month_introduced',
    'parliament_number',
    'session_number',
    'reinstated',
]
CATEGORICAL_COLS = [
    'bill_type',
    'bill_type_raw',
    'chamber',
    'party',
]
TARGET_COL = 'passed'

def _ca_xml_text(el) -> str:
    if el.tag in {'Identification', 'Label', 'MarginalNote'}:
        return ''
    if el.tag == 'AmendedText' and el.get('{http://www.w3.org/XML/1998/namespace}lang') == 'fr':
        return ''
    parts = []
    if el.tag == 'Text' and el.text:
        parts.append(el.text.strip())
    for child in el:
        parts.append(_ca_xml_text(child))
        if child.tail:
            parts.append(child.tail.strip())
    return ' '.join(p for p in parts if p)

def load_canada_texts(ca_txt_dir: Path) -> dict:
    texts = {}
    for session_dir in sorted(ca_txt_dir.iterdir()):
        if not session_dir.is_dir():
            continue
        session = session_dir.name
        for txt_file in session_dir.glob('*_english.txt'):
            bill_num = txt_file.name.replace('.pdf_english.txt', '').replace('_english.txt', '')
            texts[f'{session}/{bill_num}'] = txt_file.read_text(encoding='utf-8', errors='replace')
        for xml_file in session_dir.glob('*.xml'):
            key = f'{session}/{xml_file.stem}'
            if key in texts:
                continue
            try:
                root = ET.parse(xml_file).getroot()
                raw = _ca_xml_text(root)
                if raw:
                    texts[key] = raw
            except Exception as exc:
                print(f'Warning: {xml_file.name}: {exc}')
    return texts

_SKIP_TAGS = {'metadata', 'dublinCore', 'form', 'distribution-code', 'congress', 'session', 'current-chamber', 'action', 'action-date', 'action-desc', 'legis-type', 'enum', 'external-xref', 'quote'}
_PREFIX_TO_TYPE = {'HB': 'hr', 'SB': 's', 'HR': 'hres', 'SR': 'sres', 'HJR': 'hjres', 'SJR': 'sjres', 'HCR': 'hconres', 'SCR': 'sconres'}
_TYPE_TO_PREFIX = {v: k for k, v in _PREFIX_TO_TYPE.items()}

def _xml_text(el) -> str:
    if el.tag in _SKIP_TAGS:
        return ''
    parts = [el.text or '']
    for child in el:
        parts.append(_xml_text(child))
        parts.append(child.tail or '')
    return ' '.join(p.strip() for p in parts if p.strip())

def load_us_texts(us_txt_dir: Path) -> dict:
    texts = {}
    for zip_path in sorted(us_txt_dir.glob('*.zip')):
        parts = zip_path.stem.split('_')
        congress = parts[0]
        xml_type = parts[-1]
        legiscan_prefix = _TYPE_TO_PREFIX.get(xml_type)
        if legiscan_prefix is None:
            continue
        try:
            with zipfile.ZipFile(zip_path) as zf:
                for name in zf.namelist():
                    if not name.endswith('.xml'):
                        continue
                    match = re.match(rf'BILLS-{congress}{xml_type}(\d+)', name, re.IGNORECASE)
                    if not match:
                        continue
                    key = f'{congress}/{legiscan_prefix}{int(match.group(1))}'
                    try:
                        root = ET.fromstring(zf.read(name))
                        body = root.find('./legis-body')
                        if body is not None:
                            texts[key] = _xml_text(body)
                    except Exception:
                        pass
        except Exception as exc:
            print(f'Warning: {zip_path.name}: {exc}')
    return texts

ca_texts = load_canada_texts(CA_TXT)
us_texts = load_us_texts(US_TXT)

bills['full_text'] = ''
mask_ca = bills['source'] == 'canada'
mask_us = bills['source'] == 'us'
bills.loc[mask_ca, 'full_text'] = bills[mask_ca].apply(lambda r: ca_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)
bills.loc[mask_us, 'full_text'] = bills[mask_us].apply(lambda r: us_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)

print(f"Bills with full text: {(bills['full_text'].str.len() > 0).sum():,} / {len(bills):,}")
print(bills.groupby('source')['full_text'].apply(lambda s: (s.str.len() > 0).mean()).round(3))

Loaded 120,339 rows from bills.csv
Sources: {'us': 113231, 'canada': 7108}


KeyboardInterrupt: 

## 2. Prepare Metadata and Text for Late Fusion

This stage keeps the tabular and text branches separate until the end: categorical fields become embedding indices, numeric fields become a projected float vector, and text stays as raw strings for BERT chunking.

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce').fillna(0).astype(int)
    if 'introduced_date' in df.columns:
        df['introduced_date'] = pd.to_datetime(df['introduced_date'], errors='coerce')
    else:
        df['introduced_date'] = pd.NaT
    title = df.get('title', '').fillna('').astype(str)
    desc = df.get('description', '').fillna('').astype(str)
    df['title_desc'] = (title + ' ' + desc).str.strip()
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].fillna('').astype(str)
    df['_row_id'] = np.arange(len(df), dtype=int)
    return df

bills = clean_dataframe(bills)

# Build categorical vocabularies for embeddings.
cat_maps = {}
cat_cardinalities = []
for col in CATEGORICAL_COLS:
    values = ['<PAD>', '<UNK>'] + sorted(v for v in bills[col].fillna('').astype(str).unique().tolist() if v)
    mapping = {v: i for i, v in enumerate(values)}
    cat_maps[col] = mapping
    cat_cardinalities.append(len(values))

# Numeric normalization stats from the full table.
num_means = bills[NUMERIC_COLS].mean(numeric_only=True)
num_stds = bills[NUMERIC_COLS].std(numeric_only=True).replace(0, 1).fillna(1)

def encode_row_metadata(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    cat_arr = np.stack([df[col].fillna('').astype(str).map(lambda x, m=cat_maps[col]: m.get(x, m['<UNK>'])).to_numpy() for col in CATEGORICAL_COLS], axis=1)
    num_arr = ((df[NUMERIC_COLS].fillna(num_means) - num_means) / num_stds).astype('float32').to_numpy()
    return cat_arr.astype('int64'), num_arr.astype('float32')

cat_x, num_x = encode_row_metadata(bills)
texts = bills['full_text'].fillna('').astype(str).tolist()
labels = bills[TARGET_COL].to_numpy(dtype='float32')
print(f'Categorical array shape: {cat_x.shape}')
print(f'Numeric array shape: {num_x.shape}')
print(f'Label positive rate: {labels.mean():.4f}')

## 3. Tokenization, Chunking, and Dataset

Bills can exceed BERT’s token limit, so the notebook uses overlapping chunks and pools chunk-level CLS embeddings into one document vector.

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_text(text: str, tokenizer, max_length: int = MAX_LENGTH, stride: int = CHUNK_STRIDE):
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = enc['input_ids']
    attention_mask = enc['attention_mask']
    return input_ids, attention_mask

class BillLateFusionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, texts: list[str], cat_x: np.ndarray, num_x: np.ndarray, labels: np.ndarray):
        self.df = df.reset_index(drop=True)
        self.texts = texts
        self.cat_x = cat_x
        self.num_x = num_x
        self.labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'categorical': torch.tensor(self.cat_x[idx], dtype=torch.long),
            'numerical': torch.tensor(self.num_x[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32),
        }

def collate_batch(batch):
    texts = [item['text'] for item in batch]
    cats = torch.stack([item['categorical'] for item in batch])
    nums = torch.stack([item['numerical'] for item in batch])
    labels = torch.stack([item['label'] for item in batch])

    chunked_ids = []
    chunked_masks = []
    max_chunks = 1
    for text in texts:
        input_ids, attention_mask = chunk_text(text, tokenizer)
        input_ids = input_ids[:MAX_CHUNKS]
        attention_mask = attention_mask[:MAX_CHUNKS]
        chunked_ids.append(input_ids)
        chunked_masks.append(attention_mask)
        max_chunks = max(max_chunks, input_ids.size(0))

    seq_len = chunked_ids[0].size(-1)
    padded_ids = []
    padded_masks = []
    for input_ids, attention_mask in zip(chunked_ids, chunked_masks):
        if input_ids.size(0) < max_chunks:
            pad_shape = (max_chunks - input_ids.size(0), seq_len)
            id_pad = torch.zeros(pad_shape, dtype=torch.long)
            mask_pad = torch.zeros(pad_shape, dtype=torch.long)
            input_ids = torch.cat([input_ids, id_pad], dim=0)
            attention_mask = torch.cat([attention_mask, mask_pad], dim=0)
        padded_ids.append(input_ids)
        padded_masks.append(attention_mask)

    return {
        'input_ids': torch.stack(padded_ids),
        'attention_mask': torch.stack(padded_masks),
        'categorical': cats,
        'numerical': nums,
        'labels': labels,
    }

dataset = BillLateFusionDataset(bills, texts, cat_x, num_x, labels)
sample = collate_batch([dataset[0], dataset[1]])
print({k: tuple(v.shape) for k, v in sample.items()})

## 4. Build Train and Validation Loaders

This notebook starts with a simple random split so the late-fusion pipeline can be smoke-tested end to end before wiring in country-based temporal splits.

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

## 5. Late Fusion Model

The model uses the final CLS token from BERT for text, embeddings for categorical fields, a linear projection for numeric fields, and an MLP classifier on the fused vector.

In [ ]:
class LateFusionBillClassifier(nn.Module):
    def __init__(
        self,
        model_name: str,
        cat_cardinalities: list[int],
        num_numeric: int,
        bert_hidden_size: int = 768,
        cat_embed_dim: int = 16,
        num_proj_dim: int = 64,
        mlp_hidden_dims: list[int] | None = None,
        dropout: float = 0.2,
        freeze_bert: bool = True,
    ):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden_size = getattr(self.bert.config, 'hidden_size', bert_hidden_size)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.cat_embeddings = nn.ModuleList([nn.Embedding(cardinality, cat_embed_dim) for cardinality in cat_cardinalities])
        self.num_projection = nn.Linear(num_numeric, num_proj_dim)

        fusion_dim = bert_hidden_size + (len(cat_cardinalities) * cat_embed_dim) + num_proj_dim
        mlp_hidden_dims = mlp_hidden_dims or [256, 128]

        layers = []
        in_dim = fusion_dim
        for hidden_dim in mlp_hidden_dims:
            layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        self.classifier = nn.Sequential(*layers)

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        batch_size, num_chunks, seq_len = input_ids.shape
        flat_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_mask = attention_mask.view(batch_size * num_chunks, seq_len)
        outputs = self.bert(input_ids=flat_ids, attention_mask=flat_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)
        doc_embeddings = cls_embeddings.mean(dim=1)
        return doc_embeddings

    def encode_tabular(self, categorical: torch.Tensor, numerical: torch.Tensor) -> torch.Tensor:
        cat_parts = []
        for idx, embedding in enumerate(self.cat_embeddings):
            cat_parts.append(embedding(categorical[:, idx]))
        cat_vector = torch.cat(cat_parts, dim=1) if cat_parts else categorical.new_zeros((categorical.size(0), 0), dtype=torch.float32)
        num_vector = self.num_projection(numerical)
        return torch.cat([cat_vector, num_vector], dim=1)

    def forward(self, input_ids, attention_mask, categorical, numerical):
        text_vector = self.encode_text(input_ids, attention_mask)
        tabular_vector = self.encode_tabular(categorical, numerical)
        fused = torch.cat([text_vector, tabular_vector], dim=1)
        logits = self.classifier(fused).squeeze(-1)
        return logits

model = LateFusionBillClassifier(
    model_name=MODEL_NAME,
    cat_cardinalities=cat_cardinalities,
    num_numeric=len(NUMERIC_COLS),
    freeze_bert=True,
)
print(model.__class__.__name__)

## 6. Loss Function for Severe Class Imbalance

Weighted BCE is the simplest stable default. Focal loss is included as an optional alternative if the class skew remains hard to optimize.

In [ ]:
def make_pos_weight(labels: np.ndarray) -> torch.Tensor:
    positives = max(float(labels.sum()), 1.0)
    negatives = max(float(len(labels) - labels.sum()), 1.0)
    return torch.tensor([negatives / positives], dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_t * (1 - pt).pow(self.gamma) * bce
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss

pos_weight = make_pos_weight(labels)
criterion_bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion_focal = FocalLoss(alpha=0.25, gamma=2.0)
print(f'pos_weight = {pos_weight.item():.3f}')

## 7. Train / Validate Skeleton

This is a starter training loop. The next step is to wire in a country-based split strategy and run a short smoke test on a small subset before training the full model.

In [ ]:
def make_splits(df: pd.DataFrame):
    us = df[df['source'] == 'us'].sort_values(['introduced_date', '_row_id']).reset_index(drop=True)
    ca = df[df['source'] == 'canada'].sort_values(['introduced_date', '_row_id']).reset_index(drop=True)
    return us, ca

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    grad_accum_steps=1,
    use_amp=False,
    scaler=None,
    max_batches=None,
    show_progress=False,
    progress_desc='Train',
):
    model.train()
    total_loss = 0.0
    total_examples = 0
    optimizer.zero_grad(set_to_none=True)

    iterator = loader
    if show_progress:
        iterator = tqdm(loader, desc=progress_desc, leave=False, dynamic_ncols=True)

    for step, batch in enumerate(iterator, start=1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        categorical = batch['categorical'].to(device)
        numerical = batch['numerical'].to(device)
        labels = batch['labels'].to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids, attention_mask, categorical, numerical)
            raw_loss = criterion(logits, labels)
            loss = raw_loss / grad_accum_steps

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        do_step = (step % grad_accum_steps == 0) or (step == len(loader))
        if do_step:
            if use_amp and scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item() * labels.size(0)
        total_examples += labels.size(0)

        if show_progress and hasattr(iterator, 'set_postfix'):
            iterator.set_postfix(loss=f"{raw_loss.item():.4f}")

        if max_batches is not None and step >= max_batches:
            break

    return total_loss / max(total_examples, 1)

def smoke_test_batch():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    batch = sample
    with torch.no_grad():
        logits = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['categorical'].to(device),
            batch['numerical'].to(device),
        )
    print('Smoke test logits shape:', tuple(logits.shape))

smoke_test_batch()

## 8. Evaluation and Training Utilities

Defines reusable helpers for validation predictions, metric computation, threshold tuning, token-level attribution, and tabular permutation importance.

In [ ]:
def predict_loader(model, loader, device):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device),
                batch['categorical'].to(device),
                batch['numerical'].to(device),
            )
            all_logits.append(logits.detach().cpu())
            all_labels.append(batch['labels'].detach().cpu())
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))
    return labels, probs, logits

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }

def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5
    f1_vals = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1_vals))])

def score_single(model, input_ids, attention_mask, categorical, numerical, device):
    model.eval()
    with torch.no_grad():
        logits = model(
            input_ids.to(device),
            attention_mask.to(device),
            categorical.to(device),
            numerical.to(device),
        )
        probs = torch.sigmoid(logits)
    return float(probs.squeeze().cpu().item())

def top_token_importance_for_sample(model, dataset, sample_index, tokenizer, device, top_k=12):
    item = dataset[sample_index]
    batch = collate_batch([item])
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    categorical = batch['categorical']
    numerical = batch['numerical']

    base_prob = score_single(model, input_ids, attention_mask, categorical, numerical, device)
    ids = input_ids[0, 0].clone()
    mask = attention_mask[0, 0].clone()
    valid_positions = torch.where(mask == 1)[0].tolist()

    special_ids = {tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id}
    replace_id = tokenizer.mask_token_id if tokenizer.mask_token_id is not None else tokenizer.unk_token_id

    rows = []
    for pos in valid_positions:
        tok_id = int(ids[pos].item())
        if tok_id in special_ids:
            continue
        perturbed_ids = input_ids.clone()
        perturbed_ids[0, 0, pos] = replace_id
        pert_prob = score_single(model, perturbed_ids, attention_mask, categorical, numerical, device)
        token_str = tokenizer.convert_ids_to_tokens([tok_id])[0]
        rows.append({'position': pos, 'token': token_str, 'delta_prob': base_prob - pert_prob})

    imp = pd.DataFrame(rows).sort_values('delta_prob', ascending=False)
    helpful = imp.head(top_k).reset_index(drop=True)
    harmful = imp.tail(top_k).sort_values('delta_prob', ascending=True).reset_index(drop=True)
    return base_prob, helpful, harmful

def build_subset_loader(base_dataset, idx_list, cat_array, num_array, batch_size=32):
    sub_df = base_dataset.df.iloc[idx_list].reset_index(drop=True)
    sub_texts = [base_dataset.texts[i] for i in idx_list]
    sub_labels = base_dataset.labels[idx_list]
    sub_ds = BillLateFusionDataset(sub_df, sub_texts, cat_array, num_array, sub_labels)
    return torch.utils.data.DataLoader(sub_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

def permutation_importance_tabular(model, base_dataset, val_subset, metric_fn, device):
    val_idx = np.array(val_subset.indices)
    base_cat = base_dataset.cat_x[val_idx].copy()
    base_num = base_dataset.num_x[val_idx].copy()

    base_loader = build_subset_loader(base_dataset, val_idx.tolist(), base_cat, base_num, batch_size=BATCH_SIZE)
    y_true, y_prob, _ = predict_loader(model, base_loader, device)
    base_metric = metric_fn(y_true, y_prob)

    rows = []
    rng = np.random.default_rng(SEED)

    for j, name in enumerate(NUMERIC_COLS):
        pert_num = base_num.copy()
        rng.shuffle(pert_num[:, j])
        loader = build_subset_loader(base_dataset, val_idx.tolist(), base_cat, pert_num, batch_size=BATCH_SIZE)
        yt, yp, _ = predict_loader(model, loader, device)
        rows.append({'feature': name, 'type': 'numeric', 'metric_drop': base_metric - metric_fn(yt, yp)})

    for j, name in enumerate(CATEGORICAL_COLS):
        pert_cat = base_cat.copy()
        rng.shuffle(pert_cat[:, j])
        loader = build_subset_loader(base_dataset, val_idx.tolist(), pert_cat, base_num, batch_size=BATCH_SIZE)
        yt, yp, _ = predict_loader(model, loader, device)
        rows.append({'feature': name, 'type': 'categorical', 'metric_drop': base_metric - metric_fn(yt, yp)})

    return base_metric, pd.DataFrame(rows).sort_values('metric_drop', ascending=False).reset_index(drop=True)

print('Evaluation and interpretability helpers ready.')

## 9. Run Training, Metrics, and Feature Importance

Runs model training, prints validation performance (default and tuned thresholds), and outputs both token-level and metadata feature importance views.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.cuda.empty_cache()

model = model.to(device)
criterion = criterion_bce.to(device)

EPOCHS = 5
LR = 2e-4
TARGET_GLOBAL_BATCH = 8

effective_batch_size = 1 if device.type == 'cuda' else BATCH_SIZE
grad_accum_steps = max(1, TARGET_GLOBAL_BATCH // effective_batch_size)
use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=effective_batch_size,
    shuffle=True,
    collate_fn=collate_batch,
    pin_memory=(device.type == 'cuda'),
)
val_loader = torch.utils.data.DataLoader(
    val_ds,
    batch_size=effective_batch_size,
    shuffle=False,
    collate_fn=collate_batch,
    pin_memory=(device.type == 'cuda'),
)

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
print(f'Using device={device}, batch_size={effective_batch_size}, grad_accum_steps={grad_accum_steps}, amp={use_amp}')

history = []
epoch_iter = tqdm(range(1, EPOCHS + 1), desc='Epochs', total=EPOCHS, dynamic_ncols=True)
for epoch in epoch_iter:
    try:
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            grad_accum_steps=grad_accum_steps,
            use_amp=use_amp,
            scaler=scaler,
            show_progress=True,
            progress_desc=f'Epoch {epoch}/{EPOCHS}',
        )
    except torch.OutOfMemoryError:
        if device.type != 'cuda':
            raise
        print('CUDA OOM encountered. Retrying with stricter settings...')
        torch.cuda.empty_cache()
        effective_batch_size = 1
        grad_accum_steps = max(1, TARGET_GLOBAL_BATCH // effective_batch_size)
        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=effective_batch_size,
            shuffle=True,
            collate_fn=collate_batch,
            pin_memory=True,
        )
        val_loader = torch.utils.data.DataLoader(
            val_ds,
            batch_size=effective_batch_size,
            shuffle=False,
            collate_fn=collate_batch,
            pin_memory=True,
        )
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            grad_accum_steps=grad_accum_steps,
            use_amp=use_amp,
            scaler=scaler,
            show_progress=True,
            progress_desc=f'Epoch {epoch}/{EPOCHS} (retry)',
        )

    y_val, p_val, _ = predict_loader(model, val_loader, device)
    tuned_thr = best_f1_threshold(y_val, p_val)
    metrics_default = compute_metrics(y_val, p_val, threshold=0.5)
    metrics_tuned = compute_metrics(y_val, p_val, threshold=tuned_thr)
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        **{f'default_{k}': v for k, v in metrics_default.items() if k != 'confusion_matrix'},
        **{f'tuned_{k}': v for k, v in metrics_tuned.items() if k != 'confusion_matrix'},
    })
    if hasattr(epoch_iter, 'set_postfix'):
        epoch_iter.set_postfix(
            train_loss=f"{train_loss:.4f}",
            pr_auc=f"{metrics_default['pr_auc']:.4f}",
            f1=f"{metrics_tuned['f1']:.4f}",
        )
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f}")
    print('  Default@0.50:', {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics_default.items() if k != 'confusion_matrix'})
    print('  Tuned@F1  :', {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics_tuned.items() if k != 'confusion_matrix'})
    print('  Confusion default:', metrics_default['confusion_matrix'])
    print('  Confusion tuned  :', metrics_tuned['confusion_matrix'])

history_df = pd.DataFrame(history)
display(history_df)

# Token importance on one validation sample
val_example_idx = int(val_ds.indices[0])
base_prob, top_helpful, top_harmful = top_token_importance_for_sample(model, dataset, val_example_idx, tokenizer, device, top_k=12)
print(f'Base predicted pass probability for sample {val_example_idx}: {base_prob:.4f}')
print('\nTop tokens increasing pass probability (if removed, probability drops):')
display(top_helpful)
print('\nTop tokens decreasing pass probability (if removed, probability rises):')
display(top_harmful)

# Global metadata importance via PR-AUC drop
base_pr_auc, tabular_importance = permutation_importance_tabular(
    model, dataset, val_ds, metric_fn=lambda y, p: average_precision_score(y, p), device=device
)
print(f'Baseline validation PR-AUC: {base_pr_auc:.4f}')
display(tabular_importance)